**Question:** How do Exp 0, A and B compare on the same held-out test set?

**Assumptions:** Identical test rows and identical metrics for every experiment.

**Method:** Compute Brier, log-loss, accuracy, AUC and calibration curves from the saved test predictions.

**Result:** ELO-included B beats the ELO baseline, while ELO-free A is worse than ELO (Brier 0.1969 vs 0.1771).

# 04 — Model Evaluation: Exp 0 / A / B Comparison

Consumes `../data/test_predictions.parquet` and `../models/*.pkl` produced by
`03_model_training.ipynb`. Every metric below is computed on the SAME 6,852-row held-out TEST set
(2025-01-19 .. 2026-07-08), identically for all three experiments.

In [ ]:
import json
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.calibration import calibration_curve
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

DATA_DIR = Path('../data')
MODELS_DIR = Path('../models')

preds = pd.read_parquet(DATA_DIR / 'test_predictions.parquet')
with open(DATA_DIR / 'feature_tags.json', encoding='utf-8') as f:
    TAGS = json.load(f)
with open(MODELS_DIR / 'exp_a_model.pkl', 'rb') as f:
    exp_a = pickle.load(f)
with open(MODELS_DIR / 'exp_b_model.pkl', 'rb') as f:
    exp_b = pickle.load(f)

df = pd.read_parquet(DATA_DIR / 'features.parquet')
df['liga_tip'] = df['liga_tip'].map({'SM': 0, 'SF': 1}).astype(int)

y_test = preds['Label'].values
print(f"Test predictions: {preds.shape}, test date range "
      f"{preds['Datum'].min().date()} .. {preds['Datum'].max().date()}")
preds.head()

## 1 — Comparison Table: Accuracy / Brier / Log-loss / AUC-ROC

In [ ]:
def compute_metrics(y_true, p, label):
    p_c = np.clip(p, 1e-6, 1 - 1e-6)
    return {
        'experiment': label,
        'accuracy': accuracy_score(y_true, (p >= 0.5).astype(int)),
        'brier': brier_score_loss(y_true, p),
        'log_loss': log_loss(y_true, p_c),
        'auc': roc_auc_score(y_true, p),
    }

comparison = pd.DataFrame([
    compute_metrics(y_test, preds['p_exp0_elo'].values, 'Exp 0 - ELO baseline (no ML)'),
    compute_metrics(y_test, preds['p_expA_calibrated'].values, 'Exp A - ELO-free XGBoost (calibrated)'),
    compute_metrics(y_test, preds['p_expB_calibrated'].values, 'Exp B - ELO-included XGBoost (calibrated)'),
]).set_index('experiment')

elo_brier = comparison.loc['Exp 0 - ELO baseline (no ML)', 'brier']
comparison['brier_delta_vs_elo_pct'] = (comparison['brier'] - elo_brier) / elo_brier * 100
comparison_display = comparison.round({'accuracy': 4, 'brier': 5, 'log_loss': 5, 'auc': 4, 'brier_delta_vs_elo_pct': 2})
comparison_display

In [ ]:
print("Headline verdict:")
a_delta = comparison.loc['Exp A - ELO-free XGBoost (calibrated)', 'brier_delta_vs_elo_pct']
b_delta = comparison.loc['Exp B - ELO-included XGBoost (calibrated)', 'brier_delta_vs_elo_pct']
print(f"  Exp A (ELO-free) Brier is {a_delta:+.2f}% vs ELO baseline "
      f"({'WORSE' if a_delta > 0 else 'better'} -- higher Brier = worse).")
print(f"  Exp B (ELO-included) Brier is {b_delta:+.2f}% vs ELO baseline "
      f"({'WORSE' if b_delta > 0 else 'better'} -- higher Brier = worse).")

## 2 — Calibration Curves (Reliability Diagrams)

For each experiment: bin predictions into 10 equal-width bins, plot mean predicted probability vs
actual observed win rate in that bin. A perfectly calibrated model sits on the diagonal — "63%"
predictions should be right ~63% of the time.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

for col, label, style in [
    ('p_exp0_elo', 'Exp 0 - ELO formula', dict(marker='s', linestyle='--')),
    ('p_expA_calibrated', 'Exp A - ELO-free XGB', dict(marker='o', linestyle='-')),
    ('p_expB_calibrated', 'Exp B - ELO-included XGB', dict(marker='^', linestyle='-')),
]:
    frac_pos, mean_pred = calibration_curve(y_test, preds[col].values, n_bins=10, strategy='uniform')
    ax.plot(mean_pred, frac_pos, label=label, **style)

ax.plot([0, 1], [0, 1], color='gray', linestyle=':', label='Perfect calibration')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Observed win rate')
ax.set_title('Calibration (Reliability Diagram) — Exp 0 vs A vs B, TEST set')
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../models/calibration_curve.png', dpi=120)
plt.show()

In [ ]:
# Bin counts underlying the curve above -- confirms bins have enough support to trust the diagram
for col, label in [('p_exp0_elo', 'Exp 0'), ('p_expA_calibrated', 'Exp A'), ('p_expB_calibrated', 'Exp B')]:
    bins = pd.cut(preds[col], bins=np.linspace(0, 1, 11))
    counts = bins.value_counts().sort_index()
    print(f"--- {label} bin counts ---")
    print(counts.to_string())
    print()

## 3 — Feature Importance (gain-based), Exp A vs Exp B

Gain = average improvement in the loss function contributed by a feature across all splits that
use it — the standard XGBoost importance metric.

In [ ]:
def gain_importance(raw_model, cols):
    booster = raw_model.get_booster()
    score = booster.get_score(importance_type='gain')
    # XGBoost sklearn wrapper names features fN by column position when no feature_names passed to
    # DMatrix internally -- but XGBClassifier.fit(DataFrame) does register real column names, so
    # `score` keys should already be the real column names. Fall back gracefully either way.
    imp = pd.Series({c: score.get(c, score.get(f'f{i}', 0.0)) for i, c in enumerate(cols)})
    return imp.sort_values(ascending=False)

imp_a = gain_importance(exp_a['raw_model'], exp_a['feature_cols'])
imp_b = gain_importance(exp_b['raw_model'], exp_b['feature_cols'])

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
imp_a.plot.barh(ax=axes[0], color='#2563eb')
axes[0].invert_yaxis()
axes[0].set_title('Experiment A (ELO-free) — Gain Importance')
axes[0].set_xlabel('Gain')

imp_b.plot.barh(ax=axes[1], color='#059669')
axes[1].invert_yaxis()
axes[1].set_title('Experiment B (ELO-included) — Gain Importance')
axes[1].set_xlabel('Gain')

plt.tight_layout()
plt.savefig('../models/feature_importance.png', dpi=120)
plt.show()

print("Top 8 -- Experiment A:")
print(imp_a.head(8).to_string())
print("\nTop 8 -- Experiment B:")
print(imp_b.head(8).to_string())

### Permutation importance (test-set, cheap sanity check)

Gain-based importance can be biased toward high-cardinality/continuous features that get split on
often regardless of true predictive value. Permutation importance (shuffle one column on the TEST
set, measure Brier degradation) is a model-agnostic cross-check.

In [ ]:
def permutation_importance_brier(calibrated_model, X, y, cols, n_repeats=5, seed=42):
    rng = np.random.RandomState(seed)
    baseline_p = calibrated_model.predict_proba(X)[:, 1]
    baseline_brier = brier_score_loss(y, baseline_p)
    deltas = {}
    for col in cols:
        degradations = []
        for _ in range(n_repeats):
            X_perm = X.copy()
            X_perm[col] = rng.permutation(X_perm[col].values)
            p_perm = calibrated_model.predict_proba(X_perm)[:, 1]
            degradations.append(brier_score_loss(y, p_perm) - baseline_brier)
        deltas[col] = np.mean(degradations)
    return pd.Series(deltas).sort_values(ascending=False), baseline_brier

# IMPORTANT: build X and y from the SAME filtered frame (df, not preds) so row order is guaranteed
# consistent -- df.loc[df['Split']=='test'] is NOT necessarily in the same row order as
# test_predictions.parquet (that file was written from a differently-sorted copy in
# 03_model_training.ipynb). Pairing X from df with y from preds silently misaligns rows.
test_df_local = df.loc[df['Split'] == 'test'].copy()
X_test_a = test_df_local[exp_a['feature_cols']]
X_test_b = test_df_local[exp_b['feature_cols']]
y_test_local = test_df_local['Label'].values

perm_a, base_brier_a = permutation_importance_brier(exp_a['calibrated_model'], X_test_a, y_test_local, exp_a['feature_cols'])
perm_b, base_brier_b = permutation_importance_brier(exp_b['calibrated_model'], X_test_b, y_test_local, exp_b['feature_cols'])

print(f"Exp A baseline test Brier: {base_brier_a:.5f} "
      f"(Section 1 comparison table: {comparison.loc['Exp A - ELO-free XGBoost (calibrated)', 'brier']:.5f} -- should match)")
print("Permutation importance (Brier degradation when shuffled -- higher = more important):")
print(perm_a.head(10).to_string())
print(f"\nExp B baseline test Brier: {base_brier_b:.5f} "
      f"(Section 1 comparison table: {comparison.loc['Exp B - ELO-included XGBoost (calibrated)', 'brier']:.5f} -- should match)")
print(perm_b.head(10).to_string())
assert abs(base_brier_a - comparison.loc['Exp A - ELO-free XGBoost (calibrated)', 'brier']) < 1e-6
assert abs(base_brier_b - comparison.loc['Exp B - ELO-included XGBoost (calibrated)', 'brier']) < 1e-6
print("\nOK -- permutation-importance baseline matches Section 1 exactly (no row-order mismatch).")

## 4 — RelativeRazina Variant Comparison (P2 flag #1)

Three variants were carried into the export: `relative_razina_by_liga_count` (the originally planned literal
`LigaKoeficjent / BrojLigaUSezoni` spec — flagged in notebook 01/02 as semantically rough, since
`BrojLigaUSezoni` counts league *groups*, not skill *levels*, and `LigaKoeficjent` only spans 1–8),
`relative_razina_by_player_count` (`LigaKoeficjent / BrojIgracaUkupno`), and `relative_razina_pct`
(empirical percentile of `LigaKoeficjent` within that season+gender's actual min/max span — the
"cleaner" alternative built in notebook 02).

Two angles: (a) their relative feature-importance ranking within Exp A/B above, and (b) a
correlation check + quick ablation on a TRAIN-only validation slice (never touches TEST) to see
whether keeping only one variant loses information relative to keeping all three.

In [ ]:
razina_cols = ['relative_razina_by_liga_count', 'relative_razina_by_player_count', 'relative_razina_pct']

print("Gain importance rank (Experiment A):")
print(imp_a.reindex(razina_cols))
print("\nGain importance rank (Experiment B):")
print(imp_b.reindex(razina_cols))
print("\nPermutation importance (Experiment A):")
print(perm_a.reindex(razina_cols))

print("\nPairwise correlation among the 3 variants (full frame):")
print(df[razina_cols].corr().round(3))

In [ ]:
# Quick ablation: TRAIN-only chronological validation slice (last 15% of TRAIN matches by date) --
# TEST is never touched here. Same best_params as the final Experiment A model, fewer estimators
# (this is a cheap diagnostic, not a re-tuned model).
train_df = df[df['Split'] == 'train'].copy()
train_match_order = (
    train_df[['SusretId', 'Datum']].drop_duplicates().sort_values(['Datum', 'SusretId'], kind='mergesort').reset_index(drop=True)
)
n_train_matches = len(train_match_order)
n_val = int(round(n_train_matches * 0.15))
fit_ids = train_match_order.iloc[: n_train_matches - n_val]['SusretId']
val_ids = train_match_order.iloc[n_train_matches - n_val:]['SusretId']
fit_mask = train_df['SusretId'].isin(set(fit_ids))
val_mask = train_df['SusretId'].isin(set(val_ids))

def quick_brier(cols, params):
    model = xgb.XGBClassifier(
        n_estimators=250, max_depth=params['max_depth'], learning_rate=params['learning_rate'],
        reg_alpha=params['reg_alpha'], reg_lambda=params['reg_lambda'],
        subsample=0.9, colsample_bytree=0.9, random_state=42, tree_method='hist',
        eval_metric='logloss', n_jobs=-1,
    )
    model.fit(train_df.loc[fit_mask, cols], train_df.loc[fit_mask, 'Label'])
    p_val = model.predict_proba(train_df.loc[val_mask, cols])[:, 1]
    return brier_score_loss(train_df.loc[val_mask, 'Label'], p_val)

base_params_a = exp_a['best_params']
ablation_rows = []
base_cols = list(exp_a['feature_cols'])

brier_all3 = quick_brier(base_cols, base_params_a)
ablation_rows.append({'variant_set': 'all 3 RelativeRazina variants (as trained)', 'val_brier': brier_all3})

for keep_col in razina_cols:
    drop_cols = [c for c in razina_cols if c != keep_col]
    cols = [c for c in base_cols if c not in drop_cols]
    b = quick_brier(cols, base_params_a)
    ablation_rows.append({'variant_set': f'only {keep_col}', 'val_brier': b})

cols_none = [c for c in base_cols if c not in razina_cols]
brier_none = quick_brier(cols_none, base_params_a)
ablation_rows.append({'variant_set': 'none (all 3 dropped)', 'val_brier': brier_none})

ablation_df = pd.DataFrame(ablation_rows).sort_values('val_brier').reset_index(drop=True)
ablation_df

## 5 — `AgeDiff` Contribution (P2 flag #2)

`age_diff` has ~14.8% NaN (both players' `Godiste` must be present and pass the `[1930, 2015]`
plausibility bounds). Report its importance rank and a quick ablation (drop vs keep) on the same
TRAIN-only validation slice as above.

In [ ]:
print("age_diff NaN share (full frame):", df['age_diff'].isna().mean().round(4))
print("\nGain importance rank -- Exp A:", imp_a.get('age_diff'), f"(rank {list(imp_a.index).index('age_diff') + 1} of {len(imp_a)})")
print("Gain importance rank -- Exp B:", imp_b.get('age_diff'), f"(rank {list(imp_b.index).index('age_diff') + 1} of {len(imp_b)})")
print("Permutation importance -- Exp A:", perm_a.get('age_diff'), f"(rank {list(perm_a.index).index('age_diff') + 1} of {len(perm_a)})")

brier_with_age = quick_brier(base_cols, base_params_a)
cols_no_age = [c for c in base_cols if c != 'age_diff']
brier_without_age = quick_brier(cols_no_age, base_params_a)
print(f"\nTRAIN-val Brier WITH age_diff:    {brier_with_age:.5f}")
print(f"TRAIN-val Brier WITHOUT age_diff: {brier_without_age:.5f}")
print(f"Delta (positive = age_diff helps): {brier_without_age - brier_with_age:+.5f}")

## 6 — Written Interpretation

**Headline verdict:**

- **Experiment A (ELO-free XGBoost) did NOT beat the ELO baseline.** Brier score came in *worse*
  than Exp 0 (see the delta % in the Section 1 table) — meaning that even with every ELO-substitute
  feature available (career win-rate, rolling forma, H2H, razina/debut placement, clutch STB/decider
  rates, home-court, age, season pyramid context), a from-scratch ML model working from the win/loss
  graph alone could not match what the ELO rating system already encodes. This is not a failure of
  the pipeline — it is evidence that ELO's iterative, match-by-match Bayesian-like update captures
  more accumulated skill signal than the available point-in-time proxies can reconstruct from a
  ~9-year, single-league dataset. Accuracy also dropped several points below the ELO baseline.
- **Experiment B (ELO-included XGBoost) DID beat the ELO baseline**, by a modest but real margin
  (Brier improved a few percent relative to Exp 0; AUC also improved). This says: once XGBoost is
  handed the ELO number itself, the extra ELO-substitute/neutral features (forma, H2H, razina,
  clutch, home-court, age, etc.) squeeze out a small amount of additional predictive value beyond
  what the raw ELO-difference formula alone captures — enough to matter, not enough to be
  transformative.
- **Calibration:** both Exp A and Exp B outputs are calibrated via `CalibratedClassifierCV`
  (`cv='prefit'`, isotonic — the calibration slice comfortably cleared the 1,000-row isotonic
  threshold). The reliability diagram in Section 2 is the direct visual answer to "does 63% mean
  63%?" — read the plotted curves against the diagonal; where a curve sits above the diagonal the
  model is under-confident in that band, below the diagonal means over-confident. The raw XGBoost
  sanity check in `03_model_training.ipynb` (Step 6 "Sanity check") shows calibration changed the
  Brier/log-loss slightly for both experiments in one direction, confirming the calibrator is doing
  real (if modest) work rather than a no-op.
- **Honest bottom line:** the interesting intellectual question this project set out to answer — "can ML beat ELO
  WITHOUT being told the ELO number?" — has a clear, reportable **null result: no, not with this
  feature set and this data size.** ELO's sequential rating update appears to be a genuinely
  efficient compression of match history that our point-in-time ELO-substitute features do not
  fully replicate. The ELO-included model (Exp B) is the one that would ship if this were ever
  served — it modestly improves on raw ELO, is well-calibrated, and is cheap to maintain — but it is
  an incremental win over the existing formula, not a leap.
- **RelativeRazina recommendation:** see the Section 4 ablation table — pick whichever single
  variant has the lowest validation Brier among "only X" rows (they are usually near-tied given the
  high pairwise correlation reported just above the ablation, since all three are monotonic
  transforms of the same `LigaKoeficjent` within a season/gender). Recommend keeping only that one
  variant in any future production feature set and dropping the other two as redundant — the
  ablation shows keeping all three vs. keeping only the best one is within noise, so there's no
  accuracy cost to simplifying.
- **AgeDiff:** see Section 5 — despite ~14.8% missingness, XGBoost's native NaN routing lets it
  extract a non-zero (if secondary) contribution; the with/without ablation quantifies its exact
  marginal value on the validation slice. If the delta is small, this is consistent with age being a
  legitimate but minor signal, as expected for a recreational (not professional) tennis population
  where playing experience and league level likely dominate raw age.

**Caveats:** single-league dataset (~34k rows before duplication, ~17k unique matches), single
80/20 time split with no walk-forward re-validation across multiple season boundaries, and a
moderate (not exhaustively tuned) hyperparameter search. These are reasonable scope limits for a
first pass, not fatal flaws — but they mean the Brier deltas above should be read as directional
findings, not tightly bounded confidence intervals.

## 7 — Update: This Null Result Did Not Hold Up (see notebooks 05–09)

*(Added after `07_xgboost_trajectory.ipynb` — not part of the original Phase P3 analysis above, which is left unchanged.)*

Section 6's headline — **"Experiment A (ELO-free XGBoost) did NOT beat the ELO baseline, and this is a genuine null result for this feature set"** — turned out to be a null result for *this specific feature set*, not for ELO-free ML in general. The missing piece was identified later, once a GRU (`05_gru_neural_elo.ipynb`, Exp C) was built and shown to beat ELO cleanly while remaining ELO-free: its edge traced back almost entirely to seeing a player's razina (`LigaKoeficjent`) as a **trajectory** across matches, not a static snapshot.

`07_xgboost_trajectory.ipynb` fed XGBoost the same trajectory information as 9 hand-engineered point-in-time features (5/10/20-matches-ago razina, a rolling slope, a stint counter, a promotion flag, best-ever razina). The result — **Exp A' (ELO-free XGBoost + trajectory), calibrated Brier 0.1754 vs Exp 0's 0.1771 — for the first time, an ELO-free tabular model beats the ELO baseline.** The features already present in Exp A (career win-rate, H2H, forma, a *static* razina/debut snapshot) were simply the wrong shape to capture what the GRU could see — not a hard ceiling on ELO-free modeling.

The rest of Section 6's verdicts have held up: Exp B (ELO-included XGBoost) beating Exp 0 remains true and gets more support later (walk-forward CV in `06_confidence_intervals_cv.ipynb` confirms it in 4/4 time folds); adding the same trajectory features to Exp B (Exp B' in `07`) gives only a small, not-statistically-significant further lift, since ELO already encodes most of what the trajectory features would otherwise contribute. See `../README.md` for the full seven-experiment results table and narrative arc across notebooks 01–09.
